# GPU05 — خ۷ عضو ۲۳ (F07): NeuralProphet روی L4 (سراسری، ۴۱ سری)

> بند 7.16.1 گروه ۷-ب `doc/WBS-phase7-modeling.md`. **بیرون فهرست کوتاه رسمی
> اسپرینت C** (`doc/decisions/37-phase7-rescope.md` بند ۵: سقف F07 «حداکثر ۲ مدل») —
> افزودنش درخواست صریح کاربر بود (۲۰۲۶-۰۸-۱۶)، نه بازنگری تصمیم ۳۷.

**سؤال:** آیا معماری AR-Net + رگرسور آینده + آموزش **سراسری** روی هر ۴۱ سری
(سلف×وعده) چیزی به `lightgbm_quantile` (قهرمان فعلی، یافته‌ی ۱۵) اضافه می‌کند؟
انتظار بند 7.16: «L4 با آموزش سراسری شانس دارد» — برخلاف L1 که یافته‌ی ۲۸/۲۹
(نوت‌بوک GPU01) قبلاً نشان داد شبکه از LightGBM باخت.

⚠️ **این کد محلی هرگز اجرا نشده** (نه `torch` نه `neuralprophet` روی CPU محلی
نصب‌اند). سلول ۷-الف (R0) اولین آزمون واقعی‌اش است — اگر آن‌جا خطا داد، **ادامه
ندهید** و متن خطا را برگردانید تا `f07_neural_l4.py` اصلاح شود؛ دقیقاً همان کاری
که S0/R0 برای کل فاز ۷ طراحی شده انجام دهد.

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب/کگل از پیش نصب‌اند (نصب دوباره‌شان نمی‌خواهیم). ولی
`neuralprophet` پیش‌نصب نیست — این‌جا نصب می‌شود. نسخه‌ی دقیق در سلول ۵
(`device_report`) ثبت می‌شود.

In [ ]:
!pip install -q optuna mlflow "neuralprophet>=0.9"

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ کد اصلی داخل نوت‌بوک نوشته نمی‌شود (بند 7.8.4). `gpu_bundle.zip` را با
`python -m src.models.gpu_bundle` بسازید (نسخه‌ای که `data/external/calendar_tehran.csv`
را هم دارد — رگرسور آینده‌ی NeuralProphet از همان‌جا می‌آید) و در Drive بگذارید.

In [ ]:
MODE = "colab"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/input/phase7-bundle/gpu_bundle.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، همین‌جا می‌ایستد.** `load_l4_bridge()` همان `LevelData` سطح
L1 است (منبع/هش یکسان با GPU01) با برچسب سطح `"L4"` — چون یادگیری روی پنل L4
اتفاق می‌افتد ولی ارزیابی روی همان ردیف‌های L1 (بند 7.1.2).

In [ ]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.families.f07_neural_l4 import load_l4_bridge
data = load_l4_bridge()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

## سلول ۴ — بذر تصادفی سراسری

قطعیت کامل روی GPU تضمین‌شدنی نیست — قاعده‌ی سه seed (A7) در مرحله‌ی قهرمان
اجرا و پراکندگی‌اش گزارش می‌شود.

In [ ]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌ها فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [ ]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن باشد.

In [ ]:
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "colab"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

## سلول ۷-الف — R0: آزمایش دود

بند 7.3.2: یک برازش با هایپرپارامتر پیش‌فرض (`n_lags=7`، `epochs=30`، بدون لایه‌ی
مخفی) — فقط برای اثبات اجراپذیری. سیم‌چین نشتی بند 7.9.2 هم فعال است.

**اگر این سلول خطا داد، ادامه ندهید** — متن خطا را برگردانید تا کد اصلاح شود؛
بقیه‌ی نوت‌بوک محاسبه‌ی بی‌فایده است.

In [ ]:
from src.models.families import f07_neural_l4 as fam
from src.models.gpu_runner import smoke_test

smoke = [smoke_test(fam.FITTERS["neuralprophet_l4"], data,
                    hyperparams={"epochs": 30, "n_lags": 7})]

## سلول ۷-ب — R2: تنظیم با بودجه‌ی زمانی

Optuna TPE روی هر ۵ fold رسمی، τ=۰.۲۰. یک مدل، پس بودجه‌ی کامل روی همان یک study
می‌رود (برخلاف GPU01 که سه معماری را تقسیم می‌کرد).

In [ ]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

BUDGET_MINUTES = 45

studies = [run_gpu_study(
    fam.FITTERS["neuralprophet_l4"], SPACES["neuralprophet_l4"].fn, data,
    family=fam.FAMILY, feature_set=fam.FEATURE_SET,
    budget_minutes=BUDGET_MINUTES, compute=COMPUTE, seed=42)]

## سلول ۷-ج — قهرمان: سه seed + کالیبراسیون ACI + آزمون Diebold-Mariano

همان سه تعهد GPU01: سه seed (A7)، ACI (یافته‌ی ۲۲)، DM-test تک‌ردیفی در برابر B3
(یافته‌ی ۱۳). مدل هر fold/seed در `models/gpu/F07/neuralprophet_l4/` ذخیره می‌شود.

In [ ]:
import numpy as np
from src.models.gpu_runner import finalize_champion

best = studies[0]
print(f"مدل: {best.model_id} (pinball={best.best_pinball:.5f})\n")

champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234, 2026),
                               compute=COMPUTE, run_aci=True)]

## سلول ۷-د — راستی‌آزمایی مدل ذخیره‌شده

مدل سنگین فقط وقتی «ذخیره‌شده» حساب می‌شود که بازخوانی‌اش همان عدد را بدهد.

In [ ]:
from pathlib import Path
from src.models.axes import TUNING_TAU

stem = Path(f"models/gpu/F07/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
_, test0 = data.folds[0]
pred_reloaded = reloaded.predict(test0, TUNING_TAU)

print(f"سری‌های آموزش‌دیده: {len(reloaded.series_ids)}")
print(f"پیش‌بینی از مدل بازخوانی‌شده — میانگین={pred_reloaded.mean():.5f} "
      f"· min={pred_reloaded.min():.5f} · max={pred_reloaded.max():.5f}")
print(f"فایل‌های ذخیره‌شده: {len(list(stem.parent.glob('*')))}")
assert np.isfinite(pred_reloaded).all(), "مدل بازخوانی‌شده خروجی نامعتبر داد"
print("✅ مدل ذخیره‌شده قابل استفاده است")

## سلول ۷-ه — جدول اجباری بند 7.16.4: پیچیدگی در برابر بهره

⭐ ستون «Δ نسبت به LightGBM» عمدی است — اگر NeuralProphet نبرد، گزارش باید همین
را بنویسد، نه پنهانش کند.

In [ ]:
import json, pandas as pd
from pathlib import Path

lgbm = json.loads(Path("reports/phase7/S2_tuning_F02.json").read_text())
lgbm_pinball = lgbm["lightgbm_quantile"]["best_pinball"]

complexity_table = pd.DataFrame([{
    "مدل": best.model_id,
    "بهترین pinball": round(best.best_pinball, 5),
    "B3": round(best.baseline_b3, 5),
    "LightGBM (خ۲)": round(lgbm_pinball, 5),
    "Δ نسبت به LightGBM": round(best.best_pinball - lgbm_pinball, 5),
    "trial": best.n_trials_done,
    "ساعت-هسته": round(best.seconds / 3600, 2),
    "زمان یک برازش (R0)": round(smoke[0]["seconds"], 1),
}])
complexity_table

## سلول ۷-و — گزارش فارسی کامل

همه‌چیز روی دیسک نوشته می‌شود چون session کولب از بین می‌رود.

In [ ]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    f"جدول ۷.۱۶.۴ (پیچیدگی در برابر بهره): NeuralProphet سراسری {best.best_pinball:.5f} "
    f"در برابر LightGBM {lgbm_pinball:.5f}.",
    "بیرون فهرست کوتاه رسمی اسپرینت C — افزودنش درخواست صریح کاربر بود، نه بازنگری تصمیم ۳۷.",
    "آموزش سراسری (trend_global_local=season_global_local='global') روی هر ۴۱ سری با یک برازش — بند 7.16 «L4 با آموزش سراسری شانس دارد».",
    "پیش‌بینی چندگامه‌ی کور (نه بازخوراندن ρ واقعی) — مثل SARIMAX/ETS/Theta در f03_timeseries.py.",
    "رگرسور آینده از calendar_tehran.csv (پوشش کامل ۲۴۳ روز) نه از features_A_v1.parquet (فقط روزهای سرویس‌داده‌شده).",
]
report = render_family_report(
    "F07", "خ۷ عضو ۲۳ — NeuralProphet سراسری روی L4 (اجرای GPU)",
    studies, champions, smoke, DEVICE, notes)
save_family_report("F07", report, "F07b_neuralprophet_L4")
complexity_table.to_csv("reports/gpu/F07b_complexity_vs_gain.csv", index=False)
print(report)

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

In [ ]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F07b_neuralprophet_L4", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```